In [1]:
from pytential import sympy_pytential, min_pytential, quad_pytential
import numpy as np
from sympy import log, symbols
import plotly.graph_objects as go

This notebook explores the difference between implementing the volume / lattice constraint as a penalty method or Lagrange multipliers.

In [2]:
c0a, c1a, c0b, c1b, Va, Vb = symbols('c0a, c1a, c0b, c1b, Va, Vb')
kappa = 100000

RT = 8.134*300
fa_sym = c0a*RT*(1+log(c0a/(c0a+c1a))) + c1a*RT*(0+log(c1a/(c0a+c1a))) + (c0a+c1a)*kappa/2*(log(Va/(c0a+c1a)))**2
fa = sympy_pytential(fa_sym)
fb_sym = c0b*RT*(0+log(c0b/(c0b+c1b))) + c1b*RT*(1+log(c1b/(c0b+c1b))) + (c0b+c1b)*kappa/2*(log(Vb/(c0b+c1b)))**2
fb = sympy_pytential(fb_sym)

# Quadratic expansion

Let's now examine what happens to the quadratic expansion. Let's pick the point $c_0=.5$, $V=1$

In [3]:
y0 = {'c0': 0.5, 'c1': 0.5, 'Va': 0.500001854828482, 'Vb': 0.499998145171518, 'c0a': 0.13447138796688796, 'c0b': 0.3655286120331121, 'c1a': 0.36553046686688784, 'c1b': 0.1344695331331121}
#y0 = {'c0': 0.5, 'c1': 0.5, 'c0a':.2, 'c1a':.8, 'c0b':.8, 'c1b':.2, 'Va': 1, 'Vb': 1}
y0 = {'c0a':y0['c0a']/y0['Va'], 'c1a':y0['c1a']/y0['Va'], 'c0b':y0['c0b']/y0['Vb'], 'c1b':y0['c1b']/y0['Vb'], 'Va':1, 'Vb':1}

print(y0)

{'c0a': 0.2689417782520353, 'c1a': 0.7310582217585522, 'c0b': 0.7310599360478071, 'c1b': 0.2689400639416053, 'Va': 1, 'Vb': 1}


In [ ]:
fa_quad = quad_pytential.from_homog_pyt(fa, y0=y0)
fb_quad = quad_pytential.from_homog_pyt(fb, y0=y0)
fa_c = fa_quad.transform_to_cc_in({'c0a': 'mu0', 'c1a': 'mu1'})
fb_c = fb_quad.transform_to_cc_in({'c0b': 'mu0', 'c1b': 'mu1'})

f_c = quad_pytential.from_sympy((fa_c+fb_c).fcn_sym)
print(f_c)
f = f_c.transform_to_cc_in({'mu0': 'c0', 'mu1': 'c1'})
print(f)

f.write_to_file('f_quad')

x = ['Va', 'Vb', 'mu0', 'mu1']

f(x) = 0.5*Va*(0.268941778252036*mu0 + 0.731058221758552*mu1) + 764.42116988927*Va + 0.5*Vb*(0.731059936047809*mu0 + 0.268940063941606*mu1) + 764.42116986244*Vb + 0.5*mu0*(0.268941778252036*Va + 0.731059936047809*Vb + 0.000167211707416071*mu0 - 0.000157211690273024*mu1) + 0.00764251050541569*mu0 + 0.5*mu1*(0.731058221758552*Va + 0.268940063941606*Vb - 0.000157211690273024*mu0 + 0.000167211673129977*mu1) + 0.00764591289210164*mu1 + 5.84339726176876

f'(x)= [0.268941778252036*mu0 + 0.731058221758552*mu1 + 764.42116988927, 0.731059936047809*mu0 + 0.268940063941606*mu1 + 764.42116986244, 0.268941778252036*Va + 0.731059936047809*Vb + 0.000167211707416071*mu0 - 0.000157211690273024*mu1 + 0.00764251050541569, 0.731058221758552*Va + 0.268940063941606*Vb - 0.000157211690273024*mu0 + 0.000167211673129977*mu1 + 0.00764591289210164]

f"(x)= [[0, 0, 0.268941778252036, 0.731058221758552], [0, 0, 0.731059936047809, 0.268940063941606], [0.268941778252036, 0.731059936047

In [5]:
x_vals = np.linspace(0.01, 0.99, 50)
y_vals = np.linspace(0.01, 0.99, 50) # Renamed from y_vals to avoid confusion if y_vals is used elsewhere

# Calculate fa values: c0 = x_vals, c1 = 1-x_vals, V=1
fa_plot_vals = fa(c0a=x_vals, c1a=1-x_vals, Va=1)
fa_q_plot_vals = fa_quad(c0a=x_vals, c1a=1-x_vals, Va=1)

# Calculate fb values: c0 = x_vals, c1 = 1-x_vals, V=0
fb_plot_vals = fb(c0b=x_vals, c1b=1-x_vals, Vb=1)
fb_q_plot_vals = fb_quad(c0b=x_vals, c1b=1-x_vals, Vb=1)


# Prepare grid for the surface plot of f
X, Y = np.meshgrid(x_vals, y_vals)

f_surface_vals = f(c0=X.ravel(), c1=(1-X).ravel(), Va=Y.ravel(), Vb=(1-Y).ravel()).reshape(X.shape)


In [6]:
# Create the 3D plot
fig = go.Figure()

# Add fa line plot (V=1)
fig.add_trace(go.Scatter3d(
    x=x_vals, 
    y=np.ones_like(x_vals), 
    z=fa_plot_vals,
    mode='lines',
    name='fa(c0, 1-c0, V=1)'
))

fig.add_trace(go.Scatter3d(
    x=x_vals, 
    y=np.ones_like(x_vals), 
    z=fa_q_plot_vals,
    mode='lines',
    name='fa(c0, 1-c0, V=1)'
))

# Add fb line plot (V=0)
fig.add_trace(go.Scatter3d(
    x=x_vals, 
    y=np.zeros_like(x_vals), 
    z=fb_plot_vals,
    mode='lines',
    name='fb(c0, 1-c0, V=0)'
))

# Add fb line plot (V=0)
fig.add_trace(go.Scatter3d(
    x=x_vals, 
    y=np.zeros_like(x_vals), 
    z=fb_q_plot_vals,
    mode='lines',
    name='fb(c0, 1-c0, V=0)'
))

# Add f surface plot
fig.add_trace(go.Surface(
    x=x_vals, 
    y=y_vals, 
    z=f_surface_vals,
    name='f(c0t, 1-c0t, V)',
    colorscale='Viridis',
    colorbar=dict(title='f value')
))

# Update layout
fig.update_layout(
    title='Comparison of fa, fb, and f',
    scene=dict(
        xaxis_title='c0 or c0t',
        yaxis_title='V',
        zaxis_title='Function Value'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()